# 🎯 Exercice 2 — Intent Classification

**Durée estimée : 35 minutes**

## Objectif

Construire un **classificateur d'intention** qui détermine ce que veut
l'utilisateur *avant* de lancer la recherche sémantique. Cela permet
d'adapter le comportement du chatbot selon le type de requête.

## Pourquoi c'est nécessaire ?

Sans classification d'intention, le pipeline RAG traiterait de la même façon :
- *"Bonjour !"* → lancerait une recherche inutile dans la BDD
- *"Recommande-moi un film de zombies"* → OK, c'est pertinent
- *"Qui a réalisé Shining ?"* → recherche différente (info factuelle)
- *"Quelle est la météo ?"* → hors sujet, pas de recherche du tout

## Ce que vous allez apprendre

1. Créer un **dataset d'entraînement** de classification
2. Comparer deux approches : **TF-IDF + SVM** vs **Embeddings + SVM**
3. Évaluer les performances avec un **classification report**
4. Comprendre le lien entre intent et **system prompt**

> **Analogie IA prédictive** : c'est un problème de classification
> multi-classes classique. La seule différence est la nature des features
> (texte au lieu de variables numériques).

---
## Partie 1 — Le dataset d'entraînement

On définit 4 intentions (intents) avec des exemples pour chacune.
En production, ce dataset serait plus volumineux et enrichi au fil du temps.

In [1]:
# Dataset d'entraînement : exemples par intent
TRAINING_DATA: list[dict] = [
    # ── recommendation : l'utilisateur veut une suggestion de film ──
    {"text": "Recommande-moi un bon film d'horreur", "intent": "recommendation"},
    {"text": "Je cherche un film de zombies", "intent": "recommendation"},
    {"text": "Un film effrayant pour ce soir", "intent": "recommendation"},
    {"text": "Quel film d'horreur regarder ?", "intent": "recommendation"},
    {"text": "Suggère-moi un film avec des fantômes", "intent": "recommendation"},
    {"text": "Je veux un film qui fait vraiment peur", "intent": "recommendation"},
    {"text": "Un bon slasher à voir ?", "intent": "recommendation"},
    {"text": "Film d'horreur psychologique récent", "intent": "recommendation"},
    {"text": "Quelque chose de flippant avec des démons", "intent": "recommendation"},
    {"text": "Un film de maison hantée", "intent": "recommendation"},
    {"text": "Je veux voir un film de vampire", "intent": "recommendation"},
    {"text": "Un film d'horreur français", "intent": "recommendation"},

    # ── greeting : salutation simple ──
    {"text": "Bonjour", "intent": "greeting"},
    {"text": "Salut !", "intent": "greeting"},
    {"text": "Hello", "intent": "greeting"},
    {"text": "Hey", "intent": "greeting"},
    {"text": "Coucou", "intent": "greeting"},
    {"text": "Bonsoir", "intent": "greeting"},
    {"text": "Bonjour, comment ça va ?", "intent": "greeting"},
    {"text": "Salut, tu peux m'aider ?", "intent": "greeting"},
    {"text": "Re", "intent": "greeting"},
    {"text": "Yo", "intent": "greeting"},

    # ── question_info : question factuelle sur un film ──
    {"text": "Qui a réalisé Shining ?", "intent": "question_info"},
    {"text": "En quelle année est sorti L'Exorciste ?", "intent": "question_info"},
    {"text": "Quelle est la note de Hereditary ?", "intent": "question_info"},
    {"text": "De quoi parle le film Get Out ?", "intent": "question_info"},
    {"text": "Combien a coûté le film Alien ?", "intent": "question_info"},
    {"text": "Qui joue dans Scream ?", "intent": "question_info"},
    {"text": "C'est quoi le synopsis de Midsommar ?", "intent": "question_info"},
    {"text": "Le réalisateur de Conjuring", "intent": "question_info"},
    {"text": "Quelle est la durée de Saw ?", "intent": "question_info"},
    {"text": "La date de sortie de Nope", "intent": "question_info"},

    # ── out_of_scope : hors sujet ──
    {"text": "Quelle est la météo aujourd'hui ?", "intent": "out_of_scope"},
    {"text": "Raconte-moi une blague", "intent": "out_of_scope"},
    {"text": "Quel est le cours du Bitcoin ?", "intent": "out_of_scope"},
    {"text": "Comment faire une omelette ?", "intent": "out_of_scope"},
    {"text": "Qui est le président de la France ?", "intent": "out_of_scope"},
    {"text": "Aide-moi avec mon code Python", "intent": "out_of_scope"},
    {"text": "Traduis cette phrase en anglais", "intent": "out_of_scope"},
    {"text": "Quel est le score du match ?", "intent": "out_of_scope"},
    {"text": "Réserve-moi un restaurant", "intent": "out_of_scope"},
    {"text": "Comment aller à la gare ?", "intent": "out_of_scope"},
]

# Extraire textes et labels
texts = [d["text"] for d in TRAINING_DATA]
labels = [d["intent"] for d in TRAINING_DATA]

# Distribution des classes
from collections import Counter
dist = Counter(labels)
print("Distribution des classes :")
for intent, count in sorted(dist.items()):
    print(f"  {intent:<20} {count} exemples")
print(f"\nTotal : {len(texts)} exemples")

Distribution des classes :
  greeting             10 exemples
  out_of_scope         10 exemples
  question_info        10 exemples
  recommendation       12 exemples

Total : 42 exemples


---
## Partie 2 — Séparation train/test

Comme en IA prédictive classique, on sépare les données pour évaluer
la généralisation du modèle.

In [2]:
from sklearn.model_selection import train_test_split

X_train_texts, X_test_texts, y_train, y_test = train_test_split(
    texts, labels, test_size=0.3, random_state=42, stratify=labels
)

print(f"Train : {len(X_train_texts)} exemples")
print(f"Test  : {len(X_test_texts)} exemples")
print(f"\nDistribution test :")
for intent, count in sorted(Counter(y_test).items()):
    print(f"  {intent:<20} {count}")

Train : 29 exemples
Test  : 13 exemples

Distribution test :
  greeting             3
  out_of_scope         3
  question_info        3
  recommendation       4


---
## Partie 3 — Approche 1 : TF-IDF + SVM

### TF-IDF : la baseline classique

**TF-IDF** (Term Frequency - Inverse Document Frequency) transforme le texte
en un vecteur *creux* basé sur la fréquence des mots.

C'est l'équivalent d'un encodage one-hot amélioré : chaque mot est une
dimension, pondérée par son importance dans le document.

**Limitation** : TF-IDF ne capture pas le *sens*. "zombies" et "morts-vivants"
sont deux dimensions complètement indépendantes.

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report

# 1. Vectorisation TF-IDF
tfidf = TfidfVectorizer(max_features=500)
X_train_tfidf = tfidf.fit_transform(X_train_texts)
X_test_tfidf = tfidf.transform(X_test_texts)

print(f"Dimensions TF-IDF : {X_train_tfidf.shape[1]} features (mots)")
print(f"Matrice creuse : {X_train_tfidf.nnz} valeurs non-nulles "
      f"sur {X_train_tfidf.shape[0] * X_train_tfidf.shape[1]} possibles")

# 2. Entraînement SVM
svm_tfidf = LinearSVC(random_state=42, max_iter=1000)
svm_tfidf.fit(X_train_tfidf, y_train)

# 3. Évaluation
y_pred_tfidf = svm_tfidf.predict(X_test_tfidf)
print("\n" + "=" * 50)
print("RÉSULTATS — TF-IDF + SVM")
print("=" * 50)
print(classification_report(y_test, y_pred_tfidf, zero_division=0))

Dimensions TF-IDF : 80 features (mots)
Matrice creuse : 127 valeurs non-nulles sur 2320 possibles

RÉSULTATS — TF-IDF + SVM
                precision    recall  f1-score   support

      greeting       1.00      1.00      1.00         3
  out_of_scope       0.33      0.33      0.33         3
 question_info       0.00      0.00      0.00         3
recommendation       0.80      1.00      0.89         4

      accuracy                           0.62        13
     macro avg       0.53      0.58      0.56        13
  weighted avg       0.55      0.62      0.58        13



---
## Partie 4 — Approche 2 : Embeddings + SVM

Au lieu de TF-IDF, on utilise les embeddings de `sentence-transformers`
comme features. Le modèle d'embedding capture le **sens** du texte,
pas juste la présence des mots.

In [4]:
from sentence_transformers import SentenceTransformer

# 1. Charger le modèle (réutilisé du notebook 01)
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# 2. Encoder les textes
X_train_emb = model.encode(X_train_texts, show_progress_bar=False)
X_test_emb = model.encode(X_test_texts, show_progress_bar=False)

print(f"Dimensions embeddings : {X_train_emb.shape[1]} features (dense)")
print(f"Matrice dense : chaque valeur est un float")

# 3. Entraînement SVM
svm_emb = LinearSVC(random_state=42, max_iter=1000)
svm_emb.fit(X_train_emb, y_train)

# 4. Évaluation
y_pred_emb = svm_emb.predict(X_test_emb)
print("\n" + "=" * 50)
print("RÉSULTATS — Embeddings + SVM")
print("=" * 50)
print(classification_report(y_test, y_pred_emb, zero_division=0))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Dimensions embeddings : 384 features (dense)
Matrice dense : chaque valeur est un float

RÉSULTATS — Embeddings + SVM
                precision    recall  f1-score   support

      greeting       1.00      1.00      1.00         3
  out_of_scope       0.00      0.00      0.00         3
 question_info       0.50      1.00      0.67         3
recommendation       1.00      1.00      1.00         4

      accuracy                           0.77        13
     macro avg       0.62      0.75      0.67        13
  weighted avg       0.65      0.77      0.69        13



---
## Partie 5 — Comparaison des deux approches

Mettons les résultats côte à côte :

In [5]:
from sklearn.metrics import accuracy_score

acc_tfidf = accuracy_score(y_test, y_pred_tfidf)
acc_emb = accuracy_score(y_test, y_pred_emb)

print(f"{'Approche':<25} {'Accuracy':>10}")
print(f"{'─' * 37}")
print(f"{'TF-IDF + SVM':<25} {acc_tfidf:>10.1%}")
print(f"{'Embeddings + SVM':<25} {acc_emb:>10.1%}")
print(f"{'─' * 37}")

if acc_emb >= acc_tfidf:
    print(f"\n✅ Embeddings gagne avec +{acc_emb - acc_tfidf:.1%} d'accuracy")
else:
    print(f"\n⚠ TF-IDF gagne (dataset petit, embeddings brillent sur des données plus variées)")

print("""
Note : avec un dataset aussi petit (~40 exemples), les résultats
peuvent être proches. L'avantage des embeddings se révèle sur des
formulations inédites (pas vues à l'entraînement) — c'est la capacité
de généralisation sémantique.""")

Approche                    Accuracy
─────────────────────────────────────
TF-IDF + SVM                   61.5%
Embeddings + SVM               76.9%
─────────────────────────────────────

✅ Embeddings gagne avec +15.4% d'accuracy

Note : avec un dataset aussi petit (~40 exemples), les résultats
peuvent être proches. L'avantage des embeddings se révèle sur des
formulations inédites (pas vues à l'entraînement) — c'est la capacité
de généralisation sémantique.


---
## Partie 6 — Test de généralisation

Testons les deux modèles sur des phrases **jamais vues**, avec des
formulations très différentes du dataset d'entraînement :

In [6]:
# Phrases volontairement différentes du dataset d'entraînement
test_generalization = [
    ("J'ai envie de me faire un marathon épouvante", "recommendation"),
    ("Wesh", "greeting"),
    ("C'est qui l'acteur principal dans Paranormal Activity ?", "question_info"),
    ("Tu peux m'aider à faire mes devoirs de maths ?", "out_of_scope"),
    ("Trouve-moi un truc bien gore", "recommendation"),
    ("Bonne nuit", "greeting"),
    ("Le budget de production de It", "question_info"),
    ("Commande-moi une pizza", "out_of_scope"),
]

gen_texts = [t[0] for t in test_generalization]
gen_labels = [t[1] for t in test_generalization]

# Prédictions TF-IDF
gen_tfidf = tfidf.transform(gen_texts)
pred_tfidf = svm_tfidf.predict(gen_tfidf)

# Prédictions Embeddings
gen_emb = model.encode(gen_texts)
pred_emb = svm_emb.predict(gen_emb)

print(f"{'Phrase':<52} {'Attendu':<16} {'TF-IDF':<16} {'Embeddings':<16}")
print("─" * 100)
for i, (text, expected) in enumerate(test_generalization):
    t_ok = "✅" if pred_tfidf[i] == expected else "❌"
    e_ok = "✅" if pred_emb[i] == expected else "❌"
    print(f"{text[:50]:<52} {expected:<16} {t_ok} {pred_tfidf[i]:<13} {e_ok} {pred_emb[i]:<13}")

print(f"\nTF-IDF accuracy    : {accuracy_score(gen_labels, pred_tfidf):.0%}")
print(f"Embeddings accuracy : {accuracy_score(gen_labels, pred_emb):.0%}")

Phrase                                               Attendu          TF-IDF           Embeddings      
────────────────────────────────────────────────────────────────────────────────────────────────────
J'ai envie de me faire un marathon épouvante         recommendation   ✅ recommendation ✅ recommendation
Wesh                                                 greeting         ✅ greeting      ✅ greeting     
C'est qui l'acteur principal dans Paranormal Activ   question_info    ✅ question_info ✅ question_info
Tu peux m'aider à faire mes devoirs de maths ?       out_of_scope     ❌ question_info ✅ out_of_scope 
Trouve-moi un truc bien gore                         recommendation   ❌ out_of_scope  ✅ recommendation
Bonne nuit                                           greeting         ✅ greeting      ✅ greeting     
Le budget de production de It                        question_info    ✅ question_info ✅ question_info
Commande-moi une pizza                               out_of_scope     ✅ out_of

Les embeddings généralisent typiquement mieux car ils comprennent que
*"marathon épouvante"* est sémantiquement proche de *"recommande un film d'horreur"*,
même si aucun mot n'est commun. TF-IDF, lui, ne voit que des mots inconnus.

---
## Partie 7 — Du classifier au system prompt

Dans HorrorBot, chaque intent déclenche un **system prompt différent**.
C'est ce qui permet de spécialiser le comportement du LLM selon la situation :

In [7]:
# Mapping intent → system prompt (simplifié)
SYSTEM_PROMPTS: dict[str, str] = {
    "recommendation": (
        "Tu es un expert en films d'horreur. Utilise UNIQUEMENT le contexte "
        "fourni (résultats RAG) pour recommander des films. Ne cite jamais "
        "un film qui n'est pas dans le contexte."
    ),
    "greeting": (
        "Réponds brièvement et chaleureusement à la salutation. "
        "Invite l'utilisateur à te poser une question sur les films d'horreur."
    ),
    "question_info": (
        "Réponds factuellement à la question sur le film en utilisant "
        "UNIQUEMENT le contexte fourni. Si l'information n'est pas dans "
        "le contexte, dis-le clairement."
    ),
    "out_of_scope": (
        "Explique poliment que tu es un chatbot spécialisé dans les films "
        "d'horreur et que tu ne peux pas aider sur ce sujet. Invite "
        "l'utilisateur à poser une question sur les films d'horreur."
    ),
}


def get_system_prompt(user_query: str) -> tuple[str, str]:
    """Classifie l'intent et retourne le system prompt correspondant.

    Args:
        user_query: Message de l'utilisateur.

    Returns:
        Tuple (intent détecté, system prompt correspondant).
    """
    query_emb = model.encode([user_query])
    intent = svm_emb.predict(query_emb)[0]
    return intent, SYSTEM_PROMPTS[intent]


# Démonstration
demo_queries = [
    "Trouve-moi un bon film avec des loups-garous",
    "Salut !",
    "Qui a réalisé le film Us ?",
    "Quel temps fait-il à Paris ?",
]

for query in demo_queries:
    intent, prompt = get_system_prompt(query)
    print(f'Requête : "{query}"')
    print(f'  → Intent : {intent}')
    print(f'  → Prompt : {prompt[:80]}...')
    print()

Requête : "Trouve-moi un bon film avec des loups-garous"
  → Intent : recommendation
  → Prompt : Tu es un expert en films d'horreur. Utilise UNIQUEMENT le contexte fourni (résul...

Requête : "Salut !"
  → Intent : greeting
  → Prompt : Réponds brièvement et chaleureusement à la salutation. Invite l'utilisateur à te...

Requête : "Qui a réalisé le film Us ?"
  → Intent : question_info
  → Prompt : Réponds factuellement à la question sur le film en utilisant UNIQUEMENT le conte...

Requête : "Quel temps fait-il à Paris ?"
  → Intent : out_of_scope
  → Prompt : Explique poliment que tu es un chatbot spécialisé dans les films d'horreur et qu...



---
## Partie 8 — À vous de jouer !

### Exercice 8.1 — Enrichir le dataset

Ajoutez **3 exemples par intent** au `TRAINING_DATA`, ré-entraînez
le modèle embeddings + SVM, et vérifiez si les scores s'améliorent.

In [ ]:
# Exercice 8.1 — Ajoutez vos exemples ici
NEW_EXAMPLES: list[dict] = [
    # Ajoutez 3 exemples pour "recommendation"
    # {"text": "...", "intent": "recommendation"},

    # Ajoutez 3 exemples pour "greeting"
    # {"text": "...", "intent": "greeting"},

    # Ajoutez 3 exemples pour "question_info"
    # {"text": "...", "intent": "question_info"},

    # Ajoutez 3 exemples pour "out_of_scope"
    # {"text": "...", "intent": "out_of_scope"},
]

if NEW_EXAMPLES:
    enriched = TRAINING_DATA + NEW_EXAMPLES
    e_texts = [d["text"] for d in enriched]
    e_labels = [d["intent"] for d in enriched]

    X_tr, X_te, y_tr, y_te = train_test_split(
        e_texts, e_labels, test_size=0.3, random_state=42, stratify=e_labels
    )
    X_tr_emb = model.encode(X_tr, show_progress_bar=False)
    X_te_emb = model.encode(X_te, show_progress_bar=False)

    svm_new = LinearSVC(random_state=42, max_iter=1000)
    svm_new.fit(X_tr_emb, y_tr)
    y_pred_new = svm_new.predict(X_te_emb)

    print(f"Accuracy enrichie : {accuracy_score(y_te, y_pred_new):.1%}")
    print(classification_report(y_te, y_pred_new, zero_division=0))
else:
    print("Ajoutez vos exemples dans NEW_EXAMPLES pour lancer l'entraînement !")

### Exercice 8.2 — Trouver les faiblesses

Trouvez des phrases qui trompent le classifier. Testez des cas ambigus :

In [ ]:
# Exercice 8.2 — Testez des cas limites
cas_limites = [
    "C'est nul ton film",          # Sentiment négatif, mais quel intent ?
    "Merci beaucoup",              # Politesse, pas vraiment greeting
    "J'ai vu Scream, c'était bien, t'as quoi d'autre ?",  # Mix info + reco
    # Ajoutez vos propres cas limites :
]

for phrase in cas_limites:
    intent, prompt = get_system_prompt(phrase)
    print(f'"{phrase}"')
    print(f"  → {intent}")
    print()

<details>
<summary>💡 Réflexion</summary>

Les cas limites sont normaux — aucun classifier n'est parfait. En production,
on gère ça avec :

- Un **seuil de confiance** : si le classifier hésite, on utilise un intent par défaut
- Un **intent "fallback"** pour les cas ambigus
- Un enrichissement continu du dataset à partir des erreurs détectées en monitoring

Dans HorrorBot, le classifier DeBERTa-v3 (plus puissant que notre SVM)
atteint ~95% de précision sur un dataset plus large.
</details>

---
## Récapitulatif

Dans cet exercice, vous avez :

✅ Créé un **dataset de classification d'intentions** (4 classes)
✅ Comparé **TF-IDF vs Embeddings** comme features pour un SVM
✅ Constaté l'avantage des embeddings en **généralisation**
✅ Relié chaque intent à un **system prompt** spécifique

### Le pipeline jusqu'ici

```
Requête utilisateur
    │
    ▼
┌─────────────────────┐
│  Intent Classifier   │  ← Notebook 02 (ce notebook)
│  (Embeddings + SVM)  │
└────────┬────────────┘
         │ intent détecté
         ▼
┌─────────────────────┐
│  Recherche RAG       │  ← Notebook 01 (embeddings + cosinus)
│  (si recommendation  │
│   ou question_info)  │
└────────┬────────────┘
         │ contexte films
         ▼
┌─────────────────────┐
│  LLM + system prompt │  ← Notebook 03 (prochain !)
│  adapté à l'intent   │
└─────────────────────┘
```

### Prochaine étape

Dans le **notebook 03**, on assemble toutes les briques en un
**mini-pipeline RAG complet** qui appelle un LLM pour générer la réponse.